# 05c - Evaluación comparativa focused_chunked

Objetivos:
- comparar `base`, `enriched` y `focused_chunked`
- evaluar sólo consultas cubiertas por DBC
- medir keyword, semantic e hybrid en un experimento controlado


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import CURATED_FOCUSED_CHUNKED_PARQUET_PATH, OUTPUTS_DIR
from src.data_loader import load_evaluation_queries
from src.evaluation import (
    evaluate_hybrid_search,
    evaluate_keyword_search,
    evaluate_semantic_search,
    results_to_frame,
    summarize_results,
    summary_to_frame,
)

K = 5
OUTPUT_RESULTS = OUTPUTS_DIR / "evaluation_results_focused_chunked.csv"
OUTPUT_SUMMARY = OUTPUTS_DIR / "evaluation_summary_focused_chunked.csv"


In [2]:
focused_df = pd.read_parquet(CURATED_FOCUSED_CHUNKED_PARQUET_PATH)
focused_cuces = set(focused_df["cuce"].astype(str))

covered_query_ids_by_split = {}
coverage_rows = []
for split in ["dev", "val", "test"]:
    queries = load_evaluation_queries(split)
    covered = queries[queries["relevant_cuce"].astype(str).isin(focused_cuces)].copy()
    covered_query_ids = set(covered["query_id"].astype(str))
    covered_query_ids_by_split[split] = covered_query_ids
    coverage_rows.append({
        "split": split,
        "total_queries": len(queries),
        "covered_queries": len(covered),
        "query_ids": ", ".join(covered["query_id"].astype(str).tolist()),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
print("Covered query ids by split:")
for split, ids in covered_query_ids_by_split.items():
    print(split, sorted(ids))


,split,total_queries,covered_queries,query_ids
0,dev,6,1,dev_004
1,val,3,0,
2,test,3,1,test_003


Covered query ids by split:
dev ['dev_004']
val []
test ['test_003']


In [3]:
scenarios = [
    {"method": "keyword", "label": "keyword_base_covered", "metadata_filters": {"corpus_variant": "base"}},
    {"method": "keyword", "label": "keyword_enriched_covered", "metadata_filters": {"corpus_variant": "enriched"}},
    {"method": "keyword", "label": "keyword_focused_covered", "metadata_filters": {"corpus_variant": "focused_chunked"}},
    {"method": "semantic", "label": "semantic_base_covered", "metadata_filters": {"corpus_variant": "base"}},
    {"method": "semantic", "label": "semantic_enriched_covered", "metadata_filters": {"corpus_variant": "enriched"}},
    {"method": "semantic", "label": "semantic_focused_covered", "metadata_filters": {"corpus_variant": "focused_chunked"}},
    {"method": "hybrid", "label": "hybrid_base_covered", "metadata_filters": {"corpus_variant": "base"}},
    {"method": "hybrid", "label": "hybrid_enriched_covered", "metadata_filters": {"corpus_variant": "enriched"}},
    {"method": "hybrid", "label": "hybrid_focused_covered", "metadata_filters": {"corpus_variant": "focused_chunked"}},
]

frames = []
summary_frames = []

for split in ["dev", "val", "test"]:
    split_query_ids = covered_query_ids_by_split.get(split, set())
    if not split_query_ids:
        print(f"Sin queries cubiertas para {split}; se omite del resumen.")
        continue
    for scenario in scenarios:
        if scenario["method"] == "keyword":
            results = evaluate_keyword_search(
                split=split,
                k=K,
                metadata_filters_extra=scenario["metadata_filters"],
                method_label=scenario["label"],
                query_ids=split_query_ids,
            )
        elif scenario["method"] == "semantic":
            results = evaluate_semantic_search(
                split=split,
                k=K,
                metadata_filters_extra=scenario["metadata_filters"],
                method_label=scenario["label"],
                query_ids=split_query_ids,
            )
        else:
            results = evaluate_hybrid_search(
                split=split,
                k=K,
                metadata_filters_extra=scenario["metadata_filters"],
                method_label=scenario["label"],
                query_ids=split_query_ids,
            )

        frame = results_to_frame(results)
        frame["split"] = split
        frames.append(frame)

        summary = summary_to_frame(summarize_results(results))
        summary["split"] = split
        summary_frames.append(summary)

results_df = pd.concat(frames, ignore_index=True)
summary_df = pd.concat(summary_frames, ignore_index=True)

OUTPUT_RESULTS.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_RESULTS, index=False, encoding="utf-8-sig")
summary_df.to_csv(OUTPUT_SUMMARY, index=False, encoding="utf-8-sig")

display(summary_df.sort_values(["split", "method"]))
print("Resultados:", OUTPUT_RESULTS)
print("Resumen:", OUTPUT_SUMMARY)


Sin queries cubiertas para val; se omite del resumen.


,method,query_count,k,mean_precision_at_k,mean_recall_at_k,mean_reciprocal_rank,hit_rate_at_k,split
6,hybrid_base_covered,1,5,0.2,1.0,1.0,1.0,dev
7,hybrid_enriched_covered,1,5,0.2,1.0,1.0,1.0,dev
8,hybrid_focused_covered,1,5,0.2,1.0,1.0,1.0,dev
0,keyword_base_covered,1,5,0.2,1.0,1.0,1.0,dev
1,keyword_enriched_covered,1,5,0.2,1.0,0.5,1.0,dev
2,keyword_focused_covered,1,5,0.2,1.0,1.0,1.0,dev
3,semantic_base_covered,1,5,0.0,0.0,0.0,0.0,dev
4,semantic_enriched_covered,1,5,0.0,0.0,0.0,0.0,dev
5,semantic_focused_covered,1,5,0.0,0.0,0.0,0.0,dev
15,hybrid_base_covered,1,5,0.2,1.0,1.0,1.0,test


Resultados: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_results_focused_chunked.csv
Resumen: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_summary_focused_chunked.csv
